# Query Yahoo Fantasy Iceberg Tables

This notebook demonstrates how to load and query the Yahoo Fantasy data stored in Iceberg tables.

The notebook correctly handles Windows path resolution by:
1. Finding the project root from `pyproject.toml`
2. Using absolute paths for the Iceberg catalog
3. Using the correct warehouse location

## Setup

In [5]:
%load_ext autotime

time: 0 ns (started: 2026-07-23 17:30:21 -05:00)


In [ ]:
from nfl.yahoo_fantasy import (
    YahooWarehouseClient,
    WarehouseQueryError,
    average_scoring_by_position_by_team,
    configure_polars_display,
    enrich_weekly_team_points,
    format_table_for_display,
    league_average_by_position,
    league_team_info,
    make_table_loaders,
    player_points_health,
    standings_summary as query_standings_summary,
    weekly_team_points_resolved,
)

time: 1.72 s (started: 2026-07-23 17:30:21 -05:00)


In [ ]:
# Configure Polars display and table presentation mode
configure_polars_display(rows=50, cols=20, str_length=100)

time: 0 ns (started: 2026-07-23 17:30:22 -05:00)


In [ ]:
# Initialize query client from repository root and register metadata if needed.
try:
    client = YahooWarehouseClient.from_project_root()
except WarehouseQueryError as exc:
    raise RuntimeError(f"Warehouse client initialization failed: {exc}") from exc

report = client.ensure_registered()

project_root = client.paths.project_root
CATALOG_DB_PATH = client.paths.catalog_db_path
WAREHOUSE_PATH = client.paths.warehouse_path
CATALOG_URI = client.paths.catalog_uri
WAREHOUSE_URI = client.paths.warehouse_uri

# Anchor cwd to repo root so relative metadata paths resolve to ./warehouse instead of examples/warehouse.
import os
os.chdir(project_root)

print("✓ Warehouse query client loaded.")
print("  Project root:", project_root)
print("  CWD:", os.getcwd())
print("  Catalog URI:", CATALOG_URI)
print("  Warehouse path:", WAREHOUSE_PATH)
print("  Registration report:", report)

✓ Warehouse query client loaded.
  Project root: C:\Users\EricTruett\nfl
  CWD: C:\Users\EricTruett\nfl
  Catalog URI: sqlite:///C:/Users/EricTruett/nfl/iceberg_catalog.db
  Warehouse path: C:\Users\EricTruett\nfl\warehouse
  Registration report: RegistrationReport(normalized_rows=0, registered_tables=0, tables_before=11, tables_after=11)
time: 1.12 s (started: 2026-07-23 17:30:22 -05:00)


## Functions

In [ ]:
def show_table(df, drop_keys: bool = True):
    """Display a table, optionally dropping key/id columns and rounding floats/decimals."""
    display(format_table_for_display(df, drop_keys=drop_keys))

time: 16 ms (started: 2026-07-23 17:30:23 -05:00)


In [ ]:
# Utility functions for loading tables
_, maybe_load = make_table_loaders(client)

time: 0 ns (started: 2026-07-23 17:30:24 -05:00)


## Get Warehouse Information

In [11]:
if CATALOG_DB_PATH.exists():
    print("✓ Catalog database exists:", CATALOG_DB_PATH)
else:
    print("⚠ Catalog database does not exist yet:", CATALOG_DB_PATH)
    print("  It will be created on first write operation.")

✓ Catalog database exists: C:\Users\EricTruett\nfl\iceberg_catalog.db
time: 16 ms (started: 2026-07-23 17:30:24 -05:00)


In [12]:
# List available tables with library helpers.
namespaces = ["yahoo_common", "yhnfl"]
all_tables: dict[str, list[str]] = {}

for ns in namespaces:
    all_tables[ns] = client.list_tables(ns)

print("Available tables:\n")
for ns in namespaces:
    print(f"{ns}:")
    if all_tables[ns]:
        for t in all_tables[ns]:
            print(f"  - {t}")
    else:
        print("  (no tables found)")

Available tables:

yahoo_common:
  - yahoo_common.draft_pick
  - yahoo_common.league
  - yahoo_common.player
  - yahoo_common.scoring_rule
  - yahoo_common.stat_category
  - yahoo_common.team
  - yahoo_common.transaction
yhnfl:
  - yhnfl.matchups
  - yhnfl.player_stats_weekly
  - yhnfl.roster_entries
  - yhnfl.standings
time: 0 ns (started: 2026-07-23 17:30:24 -05:00)


## Example 1: League and Team Information

Load league and team data to see league context per team.

In [13]:
league_df = maybe_load("yahoo_common.league")
team_df = maybe_load("yahoo_common.team")

league_team = league_team_info(league_df=league_df, team_df=team_df)
print("League and Team Information:")
show_table(league_team)

League and Team Information:


season,league_name,team_name,owner_name
i64,str,str,str
2025,"""Dilly Dilly""","""DaSaints""","""Reggy"""
2025,"""Dilly Dilly""","""Kyler the Creator""","""Alexander"""
2025,"""Dilly Dilly""","""Action is the Juice""","""Eric"""
2025,"""Dilly Dilly""","""oooutzs""","""Arlo"""
2025,"""Dilly Dilly""","""Alameda All-Stars""","""Mr. T"""
2025,"""Dilly Dilly""","""Tompa Bae""","""Jesse"""
2025,"""Dilly Dilly""","""Kzoo Killaz""","""Drew"""
2025,"""Dilly Dilly""","""Raven-ous""","""Moon"""
2025,"""Dilly Dilly""","""Pukachu I choose you!""","""Christopher"""


time: 360 ms (started: 2026-07-23 17:30:24 -05:00)


## Example 2: NFL League Standings

Load standings data to rank teams by wins and points for.

In [14]:
STANDINGS_TABLE = "yhnfl.standings"
standings_df = None
league_df = maybe_load("yahoo_common.league")
team_df = maybe_load("yahoo_common.team")

try:
    standings_df = client.load_table(STANDINGS_TABLE)
except Exception as exc:
    print(f"Initial standings load failed for {STANDINGS_TABLE}: {exc}")

if standings_df is None:
    # Recover common notebook state issues by re-running registration once and retrying.
    _ = client.ensure_registered(namespaces=["yhnfl"])
    try:
        standings_df = client.load_table(STANDINGS_TABLE)
    except Exception as exc:
        print(f"Retry standings load failed for {STANDINGS_TABLE}: {exc}")

if standings_df is not None and league_df is not None and team_df is not None:
    standings_summary = query_standings_summary(
        standings_df=standings_df,
        league_df=league_df,
        team_df=team_df,
    )
    print("NFL League Standings:")
    show_table(standings_summary)
else:
    print(f"Missing required data for {STANDINGS_TABLE} standings view.")
    print("Available yhnfl tables:")
    print(client.list_tables("yhnfl"))

NFL League Standings:


season,league_name,team_name,owner_name,rank,wins,losses,points_for,points_against
i64,str,str,str,i64,i64,i64,f64,f64
2025,"""Dilly Dilly""","""Pukachu I choose you!""","""Christopher""",4,11,3,1854.52,1540.48
2025,"""Dilly Dilly""","""Action is the Juice""","""Eric""",1,10,4,1838.4,1672.76
2025,"""Dilly Dilly""","""Rome Sweet Rome""","""Kevin""",3,9,5,1986.66,1787.14
2025,"""Dilly Dilly""","""DaSaints""","""Reggy""",5,9,5,1756.88,1737.22
2025,"""Dilly Dilly""","""oooutzs""","""Arlo""",2,7,7,1705.1,1675.52
2025,"""Dilly Dilly""","""Kzoo Killaz""","""Drew""",6,6,8,1760.54,1817.88
2025,"""Dilly Dilly""","""Raven-ous""","""Moon""",10,6,8,1644.88,1730.42
2025,"""Dilly Dilly""","""Alameda All-Stars""","""Mr. T""",9,6,8,1623.92,1558.48
2025,"""Dilly Dilly""","""Tompa Bae""","""Jesse""",8,6,8,1571.74,1621.78


time: 94 ms (started: 2026-07-23 17:30:24 -05:00)


## Example 3: Weekly Fantasy Points by Team

This combines `yhnfl.player_stats_weekly` with `yhnfl.roster_entries` to summarize total fantasy points by week and team.

In [ ]:
stats_df = maybe_load("yhnfl.player_stats_weekly")
roster_df = maybe_load("yhnfl.roster_entries")
matchups_df = maybe_load("yhnfl.matchups")
league_df = maybe_load("yahoo_common.league")
team_df = maybe_load("yahoo_common.team")

weekly_points, points_source = weekly_team_points_resolved(
    stats_df=stats_df,
    roster_df=roster_df,
    matchups_df=matchups_df,
)

if points_source in {"player_stats", "matchups"}:
    weekly_points_display = enrich_weekly_team_points(
        weekly_points=weekly_points,
        league_df=league_df,
        team_df=team_df,
    )

    if points_source == "player_stats":
        print("Weekly Fantasy Points by Team (from player/roster stats):")
    else:
        print("Weekly Fantasy Points by Team (fallback from matchups.points):")
    show_table(weekly_points_display)
else:
    print("Required NFL weekly tables not found.")

Weekly Fantasy Points by Team (fallback from matchups.points):


season,week,league_name,team_name,owner_name,team_points
i64,i64,str,str,str,f64
2025,1,"""Dilly Dilly""","""DaSaints""","""Reggy""",154.76
2025,1,"""Dilly Dilly""","""Raven-ous""","""Moon""",132.18
2025,1,"""Dilly Dilly""","""Pukachu I choose you!""","""Christopher""",123.84
2025,1,"""Dilly Dilly""","""Kzoo Killaz""","""Drew""",122.72
2025,1,"""Dilly Dilly""","""Zero Dark Purdy""","""Matt""",119.58
2025,1,"""Dilly Dilly""","""Alameda All-Stars""","""Mr. T""",118.36
2025,1,"""Dilly Dilly""","""Rome Sweet Rome""","""Kevin""",111.4
2025,1,"""Dilly Dilly""","""I am the Tank""","""Nbones""",105.38
2025,1,"""Dilly Dilly""","""Kyler the Creator""","""Alexander""",102.62


time: 94 ms (started: 2026-07-23 17:37:58 -05:00)


## Example 4: Average Scoring by Position by Team

Shows average weekly scoring by position for each team in the league, grouped by team name. Flex slot (`W/R/T`) is treated as its own position (`FLEX`).

In [ ]:
# Example 4: Average scoring by position by team (team name)
# FLEX handling: W/R/T is treated as FLEX.

roster_df = maybe_load("yhnfl.roster_entries")
stats_df = maybe_load("yhnfl.player_stats_weekly")
team_df = maybe_load("yahoo_common.team")
weekly_points = maybe_load("yhnfl.matchups")

if roster_df is None or stats_df is None or team_df is None:
    print("Required tables are not available.")
else:
    avg_scoring_by_team_position = average_scoring_by_position_by_team(
        roster_df=roster_df,
        stats_df=stats_df,
        team_df=team_df,
        weekly_points=weekly_points,
    )

    print("Average scoring by position for each team (team name view):")
    show_table(avg_scoring_by_team_position)

    league_avg_by_position = league_average_by_position(avg_scoring_by_team_position)

    print("\nLeague average by position:")
    show_table(league_avg_by_position)

    health = player_points_health(stats_df=stats_df, roster_df=roster_df)
    if health["non_zero_fantasy_points"] == 0 and health.get("non_zero_roster_points", 0) == 0:
        print("\nNote: all position scoring is currently 0.0 because player-level points are missing in source tables.")
        print("Team-level weekly scoring (matchups) can still be valid while slot-level scoring is unavailable.")

Average scoring by position for each team (team name view):


season,team_name,position,avg_points_per_week,median_points_per_week,min_week_points,max_week_points,avg_starters,weeks_counted,avg_pct_of_team_points
i64,str,str,f64,f64,f64,f64,f64,u32,f64
2025,"""Action is the Juice""","""QB""",22.15,20.27,10.32,42.92,1.0,16,16.35
2025,"""Action is the Juice""","""RB""",36.01,34.6,5.6,68.0,2.0,16,26.21
2025,"""Action is the Juice""","""WR""",47.13,48.25,17.8,69.9,3.0,16,34.7
2025,"""Action is the Juice""","""TE""",8.36,6.4,1.2,19.5,1.0,16,6.35
2025,"""Action is the Juice""","""FLEX""",14.71,13.85,3.0,32.5,1.0,16,10.66
2025,"""Action is the Juice""","""DEF""",7.62,7.5,0.0,16.0,1.0,16,5.73
2025,"""Alameda All-Stars""","""QB""",16.96,15.6,4.74,33.3,1.0,17,14.85
2025,"""Alameda All-Stars""","""RB""",31.26,32.1,6.2,50.1,2.0,17,26.91
2025,"""Alameda All-Stars""","""WR""",35.38,35.8,15.5,67.3,3.0,17,29.43



League average by position:


season,position,league_avg_points_per_week,league_avg_pct_of_team_points
i64,str,f64,f64
2025,"""QB""",19.53,16.25
2025,"""RB""",31.44,25.78
2025,"""WR""",40.13,32.39
2025,"""TE""",11.52,9.58
2025,"""FLEX""",12.05,9.8
2025,"""DEF""",7.57,6.2


time: 94 ms (started: 2026-07-23 17:40:29 -05:00)
